# Multi-Agent Academic Research Assistant

This notebook implements the **Multi-Agent Academic Research Assistant** from the agent.md blueprint.

- **Tech stack**: CrewAI, Gradio
- **Flow**: User enters a research topic → Wikipedia → arXiv → Web Search → Synthesis → Report Writer → downloadable report

## 1. Install dependencies

In [8]:
!pip install -q crewai crewai-tools gradio Wikipedia-API arxiv duckduckgo-search python-dotenv

You should consider upgrading via the '/Users/calistus/programming/ai-projects/llm_engineering/week8/community_contributions/ebunilo_week_8/.venv/bin/python3 -m pip install --upgrade pip' command.


## 2. Environment (optional)

Set `OPENAI_API_KEY` (or other LLM API key) in `.env` for the CrewAI agents. For web search we use DuckDuckGo (no key required).

In [9]:
import os
from dotenv import load_dotenv
load_dotenv()

# CrewAI typically uses OPENAI_API_KEY for the default LLM
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("Warning: OPENAI_API_KEY not set. Set it in .env for full functionality.")

## 3. Custom tools

In [10]:
from crewai.tools import tool
import wikipediaapi
import arxiv
from pathlib import Path


@tool("Wikipedia summary tool")
def wikipedia_tool(query: str) -> str:
    """Retrieve a high-level summary and key concepts for a research topic from Wikipedia."""
    try:
        wiki = wikipediaapi.Wikipedia(
            user_agent="AcademicResearchAssistant/1.0",
            language="en"
        )
        page = wiki.page(query)
        if not page.exists():
            # Try first sentence as summary for disambiguation
            return f"No Wikipedia page found for '{query}'. Try a more specific or alternative term."
        summary = page.summary
        sections = ""
        for section in page.sections[:5]:  # First 5 sections
            sections += f"\n## {section.title}\n{section.text[:1500]}"
        return f"# {page.title}\n\n{summary}{sections}"
    except Exception as e:
        return f"Wikipedia error: {e}"


@tool("arXiv papers tool")
def arxiv_tool(query: str) -> str:
    """Retrieve relevant academic papers from arXiv: title, authors, abstract, publication date."""
    try:
        client = arxiv.Client()
        search = arxiv.Search(
            query=query,
            max_results=5,
            sort_by=arxiv.SortCriterion.SubmittedDate
        )
        lines = []
        for i, result in enumerate(client.results(search), 1):
            lines.append(
                f"Paper {i}:\n"
                f"Title: {result.title}\n"
                f"Authors: {', '.join(a.name for a in result.authors)}\n"
                f"Date: {result.published}\n"
                f"Abstract: {result.summary}\n"
            )
        return "\n---\n".join(lines) if lines else f"No arXiv papers found for: {query}"
    except Exception as e:
        return f"arXiv error: {e}"


@tool("Web search tool")
def web_search_tool(query: str) -> str:
    """Perform a general web search for current developments, news, and insights on a topic."""
    try:
        from duckduckgo_search import DDGS
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=8))
        if not results:
            return f"No web results for: {query}"
        lines = []
        for i, r in enumerate(results, 1):
            title = r.get("title", "")
            body = r.get("body", "")
            href = r.get("href", "")
            lines.append(f"{i}. {title}\n{body}\nSource: {href}")
        return "\n\n".join(lines)
    except Exception as e:
        return f"Web search error: {e}"


# Report writer state (filename will be set by the crew run)
_report_output_dir = Path("./reports")
_report_output_dir.mkdir(exist_ok=True)


@tool("Report writer tool")
def report_writer_tool(content: str, filename: str) -> str:
    """Write the final research report to a file (Markdown). Creates the file in the reports/ directory."""
    try:
        path = _report_output_dir / filename
        if not filename.endswith(".md"):
            path = path.with_suffix(".md")
        path.write_text(content, encoding="utf-8")
        return f"Report saved to {path.absolute()}"
    except Exception as e:
        return f"Report write error: {e}"


print("Tools defined: wikipedia_tool, arxiv_tool, web_search_tool, report_writer_tool")

Tools defined: wikipedia_tool, arxiv_tool, web_search_tool, report_writer_tool


## 4. Agents and tasks (CrewAI)

In [11]:
from crewai import Agent, Task, Crew, Process


def create_research_crew(topic: str):
    """Create agents and tasks for the research workflow."""
    
    # Wikipedia Research Agent
    wikipedia_agent = Agent(
        role="Wikipedia Research Agent",
        goal="Retrieve high-level conceptual understanding and key concepts from Wikipedia",
        backstory="You are an expert at summarizing encyclopedic knowledge and extracting key concepts.",
        tools=[wikipedia_tool],
        verbose=True,
        allow_delegation=False,
    )

    # Academic Paper Agent
    arxiv_agent = Agent(
        role="Academic Paper Agent",
        goal="Retrieve relevant peer-reviewed style academic papers from arXiv",
        backstory="You are skilled at finding and summarizing academic literature.",
        tools=[arxiv_tool],
        verbose=True,
        allow_delegation=False,
    )

    # Web Search Agent
    web_search_agent = Agent(
        role="Web Search Agent",
        goal="Gather current context, news, and real-time information from the web",
        backstory="You find recent developments, industry insights, and emerging trends.",
        tools=[web_search_tool],
        verbose=True,
        allow_delegation=False,
    )

    # Analysis & Synthesis Agent (no tools; uses context from previous tasks)
    synthesis_agent = Agent(
        role="Analysis & Synthesis Agent",
        goal="Transform collected research into structured knowledge and key insights",
        backstory="You compare sources, identify patterns and consensus, and organize information logically.",
        tools=[],
        verbose=True,
        allow_delegation=False,
    )

    # Report Writer Agent
    report_agent = Agent(
        role="Report Writer Agent",
        goal="Generate the final research report and save it to disk",
        backstory="You write clear, structured academic reports in Markdown.",
        tools=[report_writer_tool],
        verbose=True,
        allow_delegation=False,
    )

    # --- Tasks ---
    task_wikipedia = Task(
        description=f"Use wikipedia_tool to obtain a high-level summary and key concepts for: {topic}",
        agent=wikipedia_agent,
        expected_output="Topic overview, key concepts, and historical context from Wikipedia.",
    )

    task_arxiv = Task(
        description=f"Use arxiv_tool to retrieve relevant academic papers for: {topic}",
        agent=arxiv_agent,
        expected_output="List of relevant research papers with title, authors, abstract, and date.",
    )

    task_web = Task(
        description=f"Use web_search_tool to collect current developments and supplementary context for: {topic}",
        agent=web_search_agent,
        expected_output="News developments, industry insights, and recent breakthroughs from the web.",
    )

    task_synthesis = Task(
        description="""Analyze and synthesize all gathered information (Wikipedia, arXiv, web).
        Produce a structured outline with: Introduction, Background, Key Concepts, Important Research Papers,
        Recent Developments, Future Directions, and Conclusion. Do not write the full report yet—output the structured outline and key points.""",
        agent=synthesis_agent,
        expected_output="Structured outline and key points for the report.",
        context=[task_wikipedia, task_arxiv, task_web],
    )

    safe_name = "".join(c if c.isalnum() or c in " -_" else "_" for c in topic)[:50].strip().replace(" ", "_")
    report_filename = f"{safe_name}_research_report.md"

    task_report = Task(
        description=f"""Write the full research report in Markdown using the synthesis outline.
        Include: Introduction, Background, Key Concepts, Important Research Papers, Recent Developments, Future Directions, Conclusion.
        Then save the report using report_writer_tool with filename='{report_filename}'.""",
        agent=report_agent,
        expected_output="Final research report saved to file.",
        context=[task_synthesis],
    )

    crew = Crew(
        agents=[wikipedia_agent, arxiv_agent, web_search_agent, synthesis_agent, report_agent],
        tasks=[task_wikipedia, task_arxiv, task_web, task_synthesis, task_report],
        process=Process.sequential,
    )
    return crew, report_filename


print("Crew and task factory ready.")

Crew and task factory ready.


## 5. Run research (single topic)

In [12]:
def run_research(topic: str) -> str:
    """Execute the full research pipeline and return the report path and content."""
    crew, report_filename = create_research_crew(topic)
    result = crew.kickoff(inputs={})
    path = _report_output_dir / report_filename
    if path.exists():
        return path.read_text(encoding="utf-8"), str(path.absolute())
    return str(result), None


# Example (run in notebook):
# content, path = run_research("CRISPR gene editing")
# print(content[:2000])

## 6. Gradio interface

In [ ]:
import gradio as gr


def run_research_ui(topic: str):
    if not topic or not topic.strip():
        return "Please enter a research topic.", None
    try:
        content, path = run_research(topic.strip())
        return content, path if path else None
    except Exception as e:
        return f"Error: {e}", None


with gr.Blocks(title="Academic Research Assistant", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Multi-Agent Academic Research Assistant")
    gr.Markdown("Enter a research topic and click **Run Research** to generate a structured report (Wikipedia + arXiv + Web → synthesis → report).")
    with gr.Row():
        topic_in = gr.Textbox(
            label="Research topic",
            placeholder="e.g. CRISPR gene editing",
            scale=3,
        )
        run_btn = gr.Button("Run Research", variant="primary", scale=1)
    report_out = gr.Textbox(
        label="Research report",
        lines=20,
        max_lines=40,
    )
    file_out = gr.File(label="Download report")

    def on_run(topic):
        content, path = run_research_ui(topic)
        return content, path if path else None

    run_btn.click(
        fn=on_run,
        inputs=[topic_in],
        outputs=[report_out, file_out],
    )

demo.launch()

/var/folders/tt/s41y0dp56qdb3_xhpxmszqy00000gn/T/ipykernel_66303/3382927340.py:14: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Academic Research Assistant", theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Wikipedia Research Agent                                                                                │
│                                                                                                                 │
│  Task: Use wikipedia_tool to obtain a high-level summary and key concepts for: CRISPR editing                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool wikipedia_summary_tool executed with result: No Wikipedia page found for 'CRISPR editing'. Try a more specific or alternative term....
Tool wikipedia_summary_tool executed with result: # CRISPR

CRISPR (; acronym for clustered regularly interspaced short palindromic repeats) is a family of DNA sequences found in the genomes of prokaryotic organisms such as bacteria and archaea. Each...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Wikipedia Research Agent                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  CRISPR                                                                                                         │
│                                                                                                                 │
│  CRISPR (; acronym for clustered regularly interspaced short palindromic repeats) is a family of DNA sequences  │
│  found in the genomes of prokaryotic organisms such as bacteria and archaea. Each sequence within an            │
│  individual prokaryotic CRISPR is derived from a DNA fragment of a bacteriophage that had previously infected   │
│  the prokaryote or one of its ancestors. These sequences are used to detect and destroy DNA from similar        │
│  bacteriophages during subsequent infections. Hence these sequences play a key role in the antiviral (i.e.      │
│  anti-phage) defense system of prokaryotes and provide a form of heritable, acquired immunity. CRISPR is found  │
│  in approximately 50% of sequenced bacterial genomes and nearly 90% of sequenced archaea.                       │
│                                                                                                                 │
│  Cas9 (or "CRISPR-associated protein 9") is an enzyme that uses CRISPR sequences as a guide to recognize and    │
│  open up specific strands of DNA that are complementary to the CRISPR sequence. Cas9 enzymes together with      │
│  CRISPR sequences form the basis of a technology known as CRISPR-Cas9 that can be used to edit genes within     │
│  living organisms. This editing process has a wide variety of applications including basic biological           │
│  research, development of biotechnological products, and treatment of diseases. The development of the          │
│  CRISPR-Cas9 genome editing technique was recognized by the Nobel Prize in Chemistry in 2020 awarded to         │
│  Emmanuelle Charpentier and Jennifer Doudna.                                                                    │
│                                                                                                                 │
│  History                                                                                                        │
│  The CRISPR/Cas system evolved in nature as a means for bacteria to protect themselves from invading viruses    │
│  and bacteriophages by inserting pieces of their DNA into the host genome. This allowed the adaptive immune     │
│  system to respond accordingly on a subsequent infection. It was discovered in Streptococcus pyogenes and       │
│  later found across many other species.                                                                         │
│                                                                                                                 │
│  Locus structure                                                                                                │
│                                                                                                                 │
│  Mechanism                                                                                                      │
│  CRISPR-Cas immunity is a natural process of bacteria and archaea. CRISPR-Cas prevents bacteriophage            │
│  infection, conjugation and natural transformation by degrading foreign nucleic acids that enter the cell.      │
│                                                        

...

/Users/calistus/programming/ai-projects/llm_engineering/week8/community_contributions/ebunilo_week_8/.venv/lib/python3.10/site-packages/crewai/crew.py:1292: DeprecationWarning: deprecated
  and hasattr(agent, "multimodal")
/Users/calistus/programming/ai-projects/llm_engineering/week8/community_contributions/ebunilo_week_8/.venv/lib/python3.10/site-packages/crewai/crew.py:1293: DeprecationWarning: deprecated
  and getattr(agent, "multimodal", False)


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer Agent                                                                                     │
│                                                                                                                 │
│  Task: Write the full research report in Markdown using the synthesis outline.                                  │
│          Include: Introduction, Background, Key Concepts, Important Research Papers, Recent Developments,       │
│  Future Directions, Conclusion.                                                                                 │
│          Then save the report using report_writer_tool with filename='CRISPR_editing_research_report.md'.       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool report_writer_tool executed with result: Report saved to /Users/calistus/programming/ai-projects/llm_engineering/week8/community_contributions/ebunilo_week_8/reports/CRISPR_editing_research_report.md...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer Agent                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # CRISPR Gene Editing: A Comprehensive Research Report                                                         │
│                                                                                                                 │
│  ## I. Introduction                                                                                             │
│  CRISPR, or Clustered Regularly Interspaced Short Palindromic Repeats, represents a remarkable adaptive immune  │
│  system found naturally in prokaryotic genomes—specifically bacteria and archaea. It functions to protect       │
│  these organisms from bacteriophage infection by detecting and destroying foreign DNA sequences. The discovery  │
│  and adaptation of this system into the CRISPR-Cas9 technology has revolutionized gene editing, enabling        │
│  precise manipulation of DNA sequences with broad applications in biology, biotechnology, and medicine. The     │
│  significance of this breakthrough was underscored by the Nobel Prize in Chemistry in 2020 awarded to           │
│  Emmanuelle Charpentier and Jennifer Doudna for their pioneering work in developing CRISPR-Cas9 as a tool for   │
│  genome editing.                                                                                                │
│                                                                                                                 │
│  ## II. Background                                                                                              │
│  CRISPR sequences were first identified in the bacterium *Streptococcus pyogenes* and later found widespread    │
│  across various bacterial and archaeal species. Naturally, CRISPR functions by capturing fragments of invading  │
│  phage DNA and integrating these fragments as spacers within CRISPR arrays, creating a form of heritable        │
│  acquired immunity against subsequent infections by the same phages. Approximately 50% of sequenced bacterial   │
│  genomes and 90% of archaeal genomes harbor CRISPR loci, highlighting its prevalence and evolutionary           │
│  importance.                                                                                                    │
│                                                                                                                 │
│  ## III. Key Concepts                                                                                           │
│  ### CRISPR Locus Structure                                                                                     │
│  The CRISPR locus consists of several short, repetitive sequences separated by unique spacer sequences derived  │
│  from foreign DNA.                                                                                              │
│                                                                                                                 │
│  ### Cas Proteins                                                                                               │
│  CRISPR-associated (Cas) proteins, especially Cas9, serve as RNA-guided DNA endonucleases that target and       │
│  cleave complementary DNA sequences determined by the guide RNA.                                                │
│                                                                                                                 │
│  ### Mechanism of Immunity                             



╭────────────────────────────── Execution Traces ──────────────────────────────╮
│                                                                              │
│  🔍 Detailed execution traces are available!                                 │
│                                                                              │
│  View insights including:                                                    │
│    • Agent decision-making process                                           │
│    • Task execution flow and timing                                          │
│    • Tool usage details                                                      │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯
Would you like to view your execution traces? [y/N] (20s timeout): 

╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                      